# Exercise 8
1. Run the Apriori Notebook Shared by me on basket dataset using different Support and confidence values.
2. What is maximum size of rule that can be created?
3. At what Confidence value, Minimum number of rules are generated.

In [2]:
from collections import OrderedDict
from rich import print

In [3]:
# Generate all subsets using bits of number from 1 -> 2^n
# you can use bits to see the items to pick each turn
# This Function is used to generate all the subset of any set
def genSubsets(l):
    powerSetSize = 2 ** len(l)
    powerSet = []
    for i in range(1, powerSetSize):
        tempEle = []
        for j in range(len(l)):
            binFlagInd = i & (1 << j)
            if binFlagInd:
                tempEle.append(l[j])
        powerSet.append(tempEle)
    return powerSet

In [4]:
print(genSubsets([1, 2, 3, 4, 5]))

[
    [1],
    [2],
    [1, 2],
    [3],
    [1, 3],
    [2, 3],
    [1, 2, 3],
    [4],
    [1, 4],
    [2, 4],
    [1, 2, 4],
    [3, 4],
    [1, 3, 4],
    [2, 3, 4],
    [1, 2, 3, 4],
    [5],
    [1, 5],
    [2, 5],
    [1, 2, 5],
    [3, 5],
    [1, 3, 5],
    [2, 3, 5],
    [1, 2, 3, 5],
    [4, 5],
    [1, 4, 5],
    [2, 4, 5],
    [1, 2, 4, 5],
    [3, 4, 5],
    [1, 3, 4, 5],
    [2, 3, 4, 5],
    [1, 2, 3, 4, 5]
]

In [5]:
# get counts for all items in total
def initPass(txList):  # list of transactions, most possibly a dict
    allTx = [item for tx in txList for item in tx]
    allTx.sort()
    cntr = OrderedDict()
    for tx in allTx:
        cntr[tx] = cntr.get(tx, 0) + 1

    return cntr

In [6]:
T = [["1", "3", "4"], ["2", "3", "5"], ["1", "2", "3", "5"], ["1", "2", "5"]]
print(initPass(T))

OrderedDict({'1': 3, '2': 3, '3': 3, '4': 1, '5': 3})

In [7]:
# Apriori Algorithm Implementation
# assumes Fk1 is sorted
def genCandidate(Fk1):  # Fk1 indicates F(k-1), it is a list of lists
    Ck = []
    k1 = len(Fk1[0])

    # COMBINE STEP
    for i in range(len(Fk1) - 1):
        for j in range(i + 1, len(Fk1)):
            # combine each itemset with all ahead of it
            f1, f2 = Fk1[i], Fk1[j]

            if f1[: len(f1) - 1] == f2[: len(f2) - 1] and f1[-1] < f2[-1]:
                tempC = f1 + [f2[-1]]

                # PRUNING STEP
                subset = genSubsets(tempC)
                appendSts = True
                for item in subset:
                    # we dont append the new itemset if it has a subset of size n-1 which was not present in the previous Fk1
                    if len(item) == k1 and item not in Fk1:
                        appendSts = False
                        break
                if appendSts:
                    Ck.append(tempC)
    return Ck

In [ ]:
print(genCandidate(sorted([["1"], ["2"], ["3"], ["5"]])))

[['2', '3'], ['2', '5'], ['3', '5']]

In [9]:
# Checks if item is not in given itemset
def searchInT(t, candid):
    found = True
    for eachCandid in candid:
        if eachCandid not in t:
            found = False
            break

    return found


In [ ]:
def apriori(T, minSup):
    finalSet = []
    # setup initial counts
    c1 = initPass(T)
    # only keep items with min sup
    f = [[item] for item in c1.keys() if c1[item] / len(T) >= minSup]  # f1
    # keep saving items into finalset of all rules
    for item in f:
        finalSet.append(item)

    while len(f) != 0:
        # get the next set of itemsets
        Ck = genCandidate(f)
        freqDict = {}
        # get the counts for the new itemsets
        for t in T:
            for candidate in Ck:
                if searchInT(t, candidate):
                    freqDict[tuple(candidate)] = freqDict.get(tuple(candidate), 0) + 1

        # remove itemsets without min sup
        f = []
        for c in freqDict.keys():
            if freqDict[c] / len(T) >= minSup:
                f.append(list(c))

        if len(f) != 0:
            # store the itemsets in the final rules list
            f = sorted(f, key=lambda x: (len(x), *x))
            for item in f:
                finalSet.append(item)
    return finalSet

## Given dataset

In [20]:
data = pd.read_csv(
    "../../datasets/basket.csv", header=None, names=[f"col{i}" for i in range(6)]
)
data.head()

,col0,col1,col2,col3,col4,col5
0,LBE,Brooklyn,11204,NaN,NaN,NaN
1,MBE,WBE,BLACK,Cambria Heights,11411,NaN
2,MBE,BLACK,Yorktown Heights,10598,NaN,NaN
3,MBE,BLACK,Long Beach,11561,NaN,NaN
4,MBE,ASIAN,Brooklyn,11235,NaN,NaN


In [21]:
import numpy as np

python_lists = []
for l in list(data.values):
    l = list(l)
    if np.nan in l:
        l = l[: l.index(np.nan)]
        python_lists.append(l)

print(python_lists[:5])

[
    ['LBE', 'Brooklyn', '11204'],
    ['MBE', 'BLACK', 'Yorktown Heights', '10598'],
    ['MBE', 'BLACK', 'Long Beach', '11561'],
    ['MBE', 'ASIAN', 'Brooklyn', '11235'],
    ['MBE', 'ASIAN', 'New York', '10026']
]

In [22]:
sup = 100 / len(python_lists)
print(apriori(python_lists, sup))

[
    ['ASIAN'],
    ['BLACK'],
    ['Brooklyn'],
    ['HISPANIC'],
    ['MBE'],
    ['NON-MINORITY'],
    ['New York'],
    ['WBE'],
    ['ASIAN', 'MBE'],
    ['BLACK', 'MBE'],
    ['Brooklyn', 'MBE'],
    ['HISPANIC', 'MBE'],
    ['MBE', 'New York'],
    ['NON-MINORITY', 'New York'],
    ['NON-MINORITY', 'WBE'],
    ['New York', 'WBE'],
    ['NON-MINORITY', 'New York', 'WBE']
]

In [27]:
sup = 50 / len(python_lists)
print(apriori(python_lists, sup))

[
    ['ASIAN'],
    ['BLACK'],
    ['Bronx'],
    ['Brooklyn'],
    ['HISPANIC'],
    ['MBE'],
    ['NON-MINORITY'],
    ['New York'],
    ['WBE'],
    ['ASIAN', 'MBE'],
    ['ASIAN', 'New York'],
    ['BLACK', 'Brooklyn'],
    ['BLACK', 'MBE'],
    ['BLACK', 'New York'],
    ['Bronx', 'MBE'],
    ['Brooklyn', 'MBE'],
    ['Brooklyn', 'WBE'],
    ['HISPANIC', 'MBE'],
    ['HISPANIC', 'New York'],
    ['MBE', 'New York'],
    ['NON-MINORITY', 'New York'],
    ['NON-MINORITY', 'WBE'],
    ['New York', 'WBE'],
    ['ASIAN', 'MBE', 'New York'],
    ['BLACK', 'Brooklyn', 'MBE'],
    ['BLACK', 'MBE', 'New York'],
    ['HISPANIC', 'MBE', 'New York'],
    ['NON-MINORITY', 'New York', 'WBE']
]

In [ ]:
sup = 0 / len(python_lists)
max_length_rules = apriori(python_lists, sup)
max_length_rule = max([len(a) for a in max_length_rules])
print(max_length_rule)

4

In [ ]:
from pyECLAT import ECLAT

min_n_products = 2
max_length = max([len(x) for x in python_lists])

data.columns = range(data.shape[1])

my_eclat = ECLAT(data=data, verbose=True)

# fit the algorithm
rule_indices, rule_supports = my_eclat.fit(
    min_support=sup, min_combination=min_n_products, max_combination=max_length
)

100%|██████████| 728/728 [00:00<00:00, 11276.68it/s]


Combination 2 by 2


28it [00:00, 42.98it/s]


Combination 3 by 3


56it [00:01, 50.72it/s]


Combination 4 by 4


70it [00:01, 55.19it/s]


In [25]:
print(rule_supports)

{
    'BLACK & MBE': 0.30070422535211266,
    'Brooklyn & MBE': 0.11267605633802817,
    'WBE & NON-MINORITY': 0.3,
    'WBE & New York': 0.17535211267605633,
    'WBE & MBE': 0.16901408450704225,
    'ASIAN & MBE': 0.2,
    'NON-MINORITY & New York': 0.11830985915492957,
    'New York & MBE': 0.1704225352112676,
    'MBE & HISPANIC': 0.16408450704225352,
    'WBE & NON-MINORITY & New York': 0.11830985915492957
}

In [54]:
from itertools import combinations


def apriori_w_conf(T, minSup, minConf):
    finalSet = []
    # setup initial counts
    c1 = initPass(T)
    # only keep items with min sup
    f = [[item] for item in c1.keys() if c1[item] / len(T) >= minSup]  # f1
    # keep saving items into finalset of all rules
    for item in f:
        finalSet.append(item)

    old_sup_dict = None

    while len(f) != 0:
        # get the next set of itemsets
        Ck = genCandidate(f)
        freqDict = {}
        # get the counts for the new itemsets
        for t in T:
            for candidate in Ck:
                if searchInT(t, candidate):
                    freqDict[tuple(candidate)] = freqDict.get(tuple(candidate), 0) + 1

        # remove itemsets without min sup
        f = []
        for c in freqDict.keys():
            item_sup = freqDict[c] / len(T)

            if item_sup >= minSup:
                if old_sup_dict is None:
                    f.append(list(c))
                    continue
                valid = False

                # generate all possible antecedents
                for i in range(1, len(c)):
                    for antecedent in combinations(c, i):
                        antecedent = tuple(sorted(antecedent))

                        if antecedent in old_sup_dict:
                            conf = item_sup / (old_sup_dict[antecedent] / len(T))
                            if conf >= minConf:
                                valid = True
                                break
                    if valid:
                        break

                if valid:
                    f.append(list(c))

        if len(f) != 0:
            # store the itemsets in the final rules list
            f = sorted(f, key=lambda x: (len(x), *x))
            for item in f:
                finalSet.append(item)

        old_sup_dict = freqDict
    return finalSet

In [67]:
sup = 50 / len(python_lists)
conf = 0.984
print(apriori_w_conf(python_lists, sup, conf))

[
    ['ASIAN'],
    ['BLACK'],
    ['Bronx'],
    ['Brooklyn'],
    ['HISPANIC'],
    ['MBE'],
    ['NON-MINORITY'],
    ['New York'],
    ['WBE'],
    ['ASIAN', 'MBE'],
    ['ASIAN', 'New York'],
    ['BLACK', 'Brooklyn'],
    ['BLACK', 'MBE'],
    ['BLACK', 'New York'],
    ['Bronx', 'MBE'],
    ['Brooklyn', 'MBE'],
    ['Brooklyn', 'WBE'],
    ['HISPANIC', 'MBE'],
    ['HISPANIC', 'New York'],
    ['MBE', 'New York'],
    ['NON-MINORITY', 'New York'],
    ['NON-MINORITY', 'WBE'],
    ['New York', 'WBE'],
    ['BLACK', 'Brooklyn', 'MBE'],
    ['BLACK', 'MBE', 'New York'],
    ['HISPANIC', 'MBE', 'New York'],
    ['NON-MINORITY', 'New York', 'WBE']
]